# V3 Stage 1: Behavioral Sweep + Error Characterization

Single notebook to run the full behavioral sweep pipeline on any model.
Edit the `CONFIG` cell below and run all cells.

**Framing:** PI failure = model fails to retrieve the final value (near-last error, not primacy intrusion).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — Edit this cell, then run all cells below
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Model ──────────────────────────────────────────────────────────────
    "model": "Qwen/Qwen2.5-1.5B-Instruct",

    # ── Hardware ───────────────────────────────────────────────────────────
    "gpu": 0,               # Physical GPU index (None = auto-detect)
    "batch_size": 16,       # Inference batch size (halves on OOM)
    "max_new_tokens": 30,   # Max tokens to generate per trial

    # ── Grid ───────────────────────────────────────────────────────────────
    "key_levels":    [2, 3, 5, 7, 10, 15, 20, 25, 30],
    "update_levels": [5, 10, 15, 20, 30, 50, 100],

    # ── Trials ─────────────────────────────────────────────────────────────
    "trials_per_cell": 200,

    # ── Early stopping ─────────────────────────────────────────────────────
    "ci_threshold": 0.05,       # Wilson CI half-width for convergence
    "min_trials_check": 25,     # First convergence check after N trials

    # ── Saturation (three-zone) ────────────────────────────────────────────
    "near_zero": 0.015,   # acc <= 1.5% → near-zero
    "recovery": 0.06,     # acc >= 6% → reset counter
    "sat_count": 3,       # stop after 3 consecutive near-zero

    # ── Dataset ────────────────────────────────────────────────────────────
    "dataset_type": "ARBITRARY_SINGLE",
    "min_updates": 5,     # Floor artifact avoidance
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP — imports, path, model loading
# ═══════════════════════════════════════════════════════════════════════════

import sys, os, json, time, random
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# Find project root (works in SageMaker or local)
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "v3":
    PROJECT_ROOT = notebook_dir.parent
else:
    # Try to find it
    for p in [notebook_dir, notebook_dir.parent, notebook_dir.parent.parent]:
        if (p / "mechanistic_probing_v2" / "core").exists():
            PROJECT_ROOT = p
            break
    else:
        raise RuntimeError("Cannot find project root. Run from v3/ or project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

from mechanistic_probing_v2.core.dataset_configs import (
    load_dataset_config, get_eligible_categories, get_max_updates,
    generate_values_for_trial, build_interleaved_sequence, FIXED_COMPLETION_DEMOS,
)
from mechanistic_probing_v2.core.model_loader import (
    load_model_hf, clear_accelerator_cache, is_instruct_model, model_short_name,
)
from mechanistic_probing_v2.core.inference import run_batch_with_oom_fallback
from mechanistic_probing_v2.core.evaluation import classify_error, bootstrap_ci

print("Imports OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════════════════════════════

model_name = CONFIG["model"]
use_chat = is_instruct_model(model_name)
m_short = model_short_name(model_name)
SYSTEM_PROMPT = "Answer with ONLY the exact value. No explanation."

model, tokenizer, info = load_model_hf(model_name, gpu_idx=CONFIG["gpu"])
device = info.device

print(f"Model: {model_name}")
print(f"Type: {'Instruct' if use_chat else 'Base'}")
print(f"Device: {device}")
print(f"Context: {info.n_ctx}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CORE FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def shuffle_no_consecutive(items, rng, max_attempts=100):
    for _ in range(max_attempts):
        candidate = items.copy()
        rng.shuffle(candidate)
        ok = all(candidate[i]["category"] != candidate[i-1]["category"]
                 for i in range(1, len(candidate)))
        if ok:
            return candidate
    remaining = items.copy()
    rng.shuffle(remaining)
    result, last_cat = [], None
    while remaining:
        valid = [i for i, item in enumerate(remaining) if item["category"] != last_cat]
        if not valid:
            result.extend(remaining)
            break
        idx = rng.choice(valid)
        item = remaining.pop(idx)
        result.append(item)
        last_cat = item["category"]
    return result


def generate_trial(num_keys, num_updates, condition, seed):
    rng = random.Random(seed)
    dataset_type = CONFIG["dataset_type"]
    eligible = get_eligible_categories(dataset_type, min_values=num_updates)
    categories = rng.sample(eligible, min(num_keys, len(eligible)))
    test_category = categories[seed % num_keys]
    values_per_cat = generate_values_for_trial(dataset_type, categories, num_updates, rng)

    items = []
    for cat in categories:
        for val in values_per_cat[cat]:
            items.append({"category": cat, "value": val})
    items = shuffle_no_consecutive(items, rng)

    stream = "\n".join(f"{it['category']}: {it['value']}" for it in items)
    query_word = "first" if condition == "RI" else "last"
    cat_values = [it["value"] for it in items if it["category"] == test_category]
    expected = cat_values[0] if condition == "RI" else cat_values[-1]

    raw_prompt = (
        f"Read the following key-value stream. Each key gets updated multiple times.\n\n"
        f"{stream}\n\n"
        f"What was the {query_word} value of {test_category}?"
    )

    if use_chat and hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_prompt},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    elif not use_chat:
        prompt = f"{FIXED_COMPLETION_DEMOS}{stream}\nThe {query_word} value of {test_category} was:"
    else:
        prompt = raw_prompt

    return {
        "prompt": prompt, "condition": condition, "expected": expected,
        "initial_value": cat_values[0], "final_value": cat_values[-1],
        "all_values": cat_values, "test_category": test_category,
        "num_keys": num_keys, "num_updates": num_updates, "seed": seed,
    }


def get_error_detail(predicted, all_values):
    pred_lower = predicted.lower().strip()
    for idx, val in enumerate(all_values):
        v = val.lower()
        if v in pred_lower or pred_lower.startswith(v):
            return {"predicted_idx": idx,
                    "predicted_relative_pos": round(idx / max(len(all_values) - 1, 1), 4)}
    return {"predicted_idx": None, "predicted_relative_pos": None}


def wilson_half_width(n, k, z=1.96):
    if n == 0:
        return 1.0
    p = k / n
    return z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)


print("Functions defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PREFLIGHT
# ═══════════════════════════════════════════════════════════════════════════

kl = CONFIG["key_levels"]
ul = [u for u in CONFIG["update_levels"] if u >= CONFIG["min_updates"]]
ctx_limit = info.n_ctx

feasible = {}
for nk in kl:
    for nu in ul:
        eligible = get_eligible_categories(CONFIG["dataset_type"], min_values=nu)
        if len(eligible) < nk:
            print(f"  keys={nk}, updates={nu}: SKIP (pool)")
            break
        trial = generate_trial(nk, nu, "RI", seed=0)
        n_tokens = len(tokenizer.encode(trial["prompt"]))
        if n_tokens > int(ctx_limit * 0.85):
            print(f"  keys={nk}, updates={nu}: SKIP ({n_tokens} tokens)")
            break
        feasible[(nk, nu)] = n_tokens

print(f"\nFeasible cells: {len(feasible)} / {len(kl) * len(ul)}")
print(f"Grid: {len(kl)} keys × {len(ul)} updates")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN SWEEP
# ═══════════════════════════════════════════════════════════════════════════

results = {
    "model": model_name,
    "dataset_type": CONFIG["dataset_type"],
    "prompt_format": "chat_template" if use_chat else "completion_few_shot",
    "config": CONFIG,
    "cells": {},
    "start_time": datetime.now(timezone.utc).isoformat(),
}
all_trials = {}

# Saturation state
sat = {nk: {"RI": {"zero_count": 0}, "PI": {"zero_count": 0}} for nk in kl}

batch_size = CONFIG["batch_size"]
trials_per_cell = CONFIG["trials_per_cell"]
total_cells = len(feasible)
cell_idx = 0

save_dir = PROJECT_ROOT / "v3" / "results" / m_short
save_dir.mkdir(parents=True, exist_ok=True)

for nk in kl:
    for nu in ul:
        if (nk, nu) not in feasible:
            continue
        cell_key = f"{nk}_{nu}"

        # Skip if already done
        if cell_key in results["cells"]:
            cell_idx += 1
            continue

        # Saturation check
        is_sat = any(sat[nk][c]["zero_count"] >= CONFIG["sat_count"] for c in ["RI", "PI"])
        if is_sat:
            print(f"  [{cell_idx+1}/{total_cells}] {nk}k_{nu}u: SKIP (saturated)")
            cell_idx += 1
            continue

        clear_accelerator_cache(device)
        cell_start = time.time()
        cell_trials = {"RI": [], "PI": []}
        stopped_early = False

        # Pre-generate trials
        pre_trials = {}
        for cond in ["RI", "PI"]:
            pre_trials[cond] = []
            for t_idx in range(trials_per_cell):
                seed = hash((nk, nu, cond, t_idx, "v3")) % (2**31)
                pre_trials[cond].append(generate_trial(nk, nu, cond, seed))

        # Run in interleaved batches
        t_idx = 0
        while t_idx < trials_per_cell:
            for cond in ["RI", "PI"]:
                batch_end = min(t_idx + batch_size, trials_per_cell)
                batch = pre_trials[cond][t_idx:batch_end]
                if not batch:
                    continue
                prompts = [t["prompt"] for t in batch]
                answers, eff_bs = run_batch_with_oom_fallback(
                    model, tokenizer, prompts,
                    max_new_tokens=CONFIG["max_new_tokens"], device=device
                )
                if eff_bs < batch_size:
                    batch_size = eff_bs

                for trial, answer in zip(batch, answers):
                    error_type = classify_error(
                        answer, trial["expected"],
                        trial["initial_value"], trial["final_value"],
                        trial["all_values"], trial["condition"],
                    )
                    ed = get_error_detail(answer, trial["all_values"])
                    n_vals = len(trial["all_values"])
                    cell_trials[cond].append({
                        "seed": trial["seed"],
                        "condition": cond,
                        "query_word": "first" if cond == "RI" else "last",
                        "expected": trial["expected"],
                        "expected_idx": 0 if cond == "RI" else n_vals - 1,
                        "expected_relative_pos": 0.0 if cond == "RI" else 1.0,
                        "predicted": answer,
                        "correct": error_type == "correct",
                        "error_type": error_type,
                        "predicted_idx": ed["predicted_idx"],
                        "predicted_relative_pos": ed["predicted_relative_pos"],
                        "all_values": trial["all_values"],
                        "output_length": len(answer.split()),
                        "output_raw": answer,
                    })

            t_idx += batch_size

            # Convergence check
            n_done = len(cell_trials["RI"])
            if n_done >= CONFIG["min_trials_check"]:
                converged = True
                for c in ["RI", "PI"]:
                    corrects = [r["correct"] for r in cell_trials[c]]
                    hw = wilson_half_width(len(corrects), sum(corrects))
                    if hw > CONFIG["ci_threshold"]:
                        converged = False
                if converged:
                    stopped_early = True
                    break

        # Aggregate
        cell_stats = {}
        for cond in ["RI", "PI"]:
            rs = cell_trials[cond]
            corrects = [r["correct"] for r in rs]
            mean, ci_lo, ci_hi = bootstrap_ci(corrects)
            error_counts = {}
            for r in rs:
                error_counts[r["error_type"]] = error_counts.get(r["error_type"], 0) + 1
            fail_pos = [r["predicted_relative_pos"] for r in rs
                        if not r["correct"] and r["predicted_relative_pos"] is not None]
            garbage_n = sum(1 for r in rs if not r["correct"] and r["predicted_idx"] is None)
            cell_stats[cond] = {
                "accuracy": round(mean, 4), "ci_lower": round(ci_lo, 4),
                "ci_upper": round(ci_hi, 4), "n": len(corrects),
                "error_types": error_counts, "n_failures": sum(1 for c in corrects if not c),
                "n_garbage": garbage_n,
                "failure_avg_relative_pos": round(float(np.mean(fail_pos)), 4) if fail_pos else None,
            }

        ri_acc = cell_stats["RI"]["accuracy"]
        pi_acc = cell_stats["PI"]["accuracy"]
        gap = ri_acc - pi_acc
        if pi_acc > ri_acc:
            regime = "D"
        elif gap >= 0.25:
            regime = "C"
        elif gap >= 0.15:
            regime = "B"
        elif gap < 0.05:
            regime = "A"
        else:
            regime = "AB"

        results["cells"][cell_key] = {
            "num_keys": nk, "num_updates": nu, "stats": cell_stats,
            "regime": regime, "n_trials": len(cell_trials["RI"]),
            "max_trials": trials_per_cell, "stopped_early": stopped_early,
            "elapsed_sec": round(time.time() - cell_start, 1),
        }
        all_trials[cell_key] = cell_trials

        # Saturation update
        for cond in ["RI", "PI"]:
            acc = cell_stats[cond]["accuracy"]
            if acc <= CONFIG["near_zero"]:
                sat[nk][cond]["zero_count"] += 1
            elif acc >= CONFIG["recovery"]:
                sat[nk][cond]["zero_count"] = 0

        pi = cell_stats["PI"]
        pi_avg = pi.get("failure_avg_relative_pos")
        early = " (early)" if stopped_early else ""
        print(f"  [{cell_idx+1}/{total_cells}] {nk}k_{nu}u: "
              f"RI={ri_acc:.0%} PI={pi_acc:.0%} regime={regime} "
              f"garbage={pi['n_garbage']}/{pi['n_failures']} "
              f"fail_pos={pi_avg if pi_avg else 'n/a'} "
              f"n={len(cell_trials['RI'])}{early} "
              f"({results['cells'][cell_key]['elapsed_sec']:.1f}s)")

        cell_idx += 1

        # Checkpoint every 3 cells
        if cell_idx % 3 == 0:
            full_data = dict(results)
            full_data["trial_details"] = all_trials
            with open(save_dir / "stage1_checkpoint.json", "w") as f:
                json.dump(full_data, f, indent=2)
            print(f"    -> Checkpoint saved")

results["end_time"] = datetime.now(timezone.utc).isoformat()
print(f"\nDone. {cell_idx} cells completed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAVE FINAL RESULTS
# ═══════════════════════════════════════════════════════════════════════════

ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# Summary (no trial details)
summary = {k: v for k, v in results.items() if k != "cells"}
summary["cells"] = results["cells"]
summary_path = save_dir / f"stage1_sweep_{ts}.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# Full (with trial details)
full_data = dict(results)
full_data["trial_details"] = all_trials
full_path = save_dir / f"stage1_trials_{ts}.json"
with open(full_path, "w") as f:
    json.dump(full_data, f, indent=2)

print(f"Summary: {summary_path}")
print(f"Full:    {full_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SUMMARY TABLES
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"V3 STAGE 1 SUMMARY — {model_name}")
print(f"Dataset: {CONFIG['dataset_type']} | Min updates: {CONFIG['min_updates']}")
print(f"{'='*70}")

for cond in ["RI", "PI"]:
    print(f"\n--- {cond} Accuracy ---")
    print(f"{'Keys':>6}", end="")
    for nu in ul:
        print(f" {nu:>5}", end="")
    print()
    for nk in kl:
        print(f"{nk:>6}", end="")
        for nu in ul:
            ck = f"{nk}_{nu}"
            if ck in results["cells"]:
                acc = results["cells"][ck]["stats"][cond]["accuracy"]
                print(f" {acc:>5.0%}", end="")
            else:
                print(f" {'---':>5}", end="")
        print()

print(f"\n--- Regime Map ---")
print(f"{'Keys':>6}", end="")
for nu in ul:
    print(f" {nu:>5}", end="")
print()
for nk in kl:
    print(f"{nk:>6}", end="")
    for nu in ul:
        ck = f"{nk}_{nu}"
        if ck in results["cells"]:
            print(f" {results['cells'][ck]['regime']:>5}", end="")
        else:
            print(f" {'---':>5}", end="")
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PI FAILURE POSITION ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════

print("\n--- PI Failure Position Analysis ---")
print(f"{'Cell':>10} {'PI Acc':>7} {'AvgPos':>7} {'Garbage%':>9} {'Near-last%':>11}")
print("-" * 50)

for nk in kl:
    for nu in ul:
        ck = f"{nk}_{nu}"
        if ck not in results["cells"]:
            continue
        if ck not in all_trials:
            continue
        pi_stats = results["cells"][ck]["stats"]["PI"]
        pi_trials = all_trials[ck]["PI"]
        failures = [t for t in pi_trials if not t["correct"]]
        if not failures:
            continue

        garbage_pct = sum(1 for f in failures if f["predicted_idx"] is None) / len(failures)
        non_garbage = [f for f in failures if f["predicted_relative_pos"] is not None]
        avg_pos = np.mean([f["predicted_relative_pos"] for f in non_garbage]) if non_garbage else None
        near_last = sum(1 for f in non_garbage if f["predicted_relative_pos"] >= 0.7) / len(failures) if failures else 0

        print(f" {nk}k_{nu}u {pi_stats['accuracy']:>6.0%} "
              f"{avg_pos:>7.2f} " if avg_pos is not None else f" {nk}k_{nu}u {pi_stats['accuracy']:>6.0%} {'n/a':>7} ",
              end="")
        print(f"{garbage_pct:>8.0%} {near_last:>10.0%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PI FAILURE POSITION HISTOGRAM (per cell)
# ═══════════════════════════════════════════════════════════════════════════

from collections import Counter

# Pick regime B or C cells for detailed analysis
interesting = {ck: c for ck, c in results["cells"].items() if c["regime"] in ["B", "C", "AB"]}

for ck in sorted(interesting.keys()):
    if ck not in all_trials:
        continue
    cell = interesting[ck]
    pi_trials = all_trials[ck]["PI"]
    n_updates = cell["num_updates"]

    print(f"\n--- {ck} (regime={cell['regime']}, PI={cell['stats']['PI']['accuracy']:.0%}) ---")

    pos_counts = Counter()
    n_garbage = 0
    for t in pi_trials:
        if t["predicted_idx"] is not None:
            pos_counts[t["predicted_idx"]] += 1
        elif not t["correct"]:
            n_garbage += 1

    n_total = len(pi_trials)
    for pos in range(n_updates):
        count = pos_counts.get(pos, 0)
        bar = "#" * min(count, 40)
        tag = " ← FIRST" if pos == 0 else (" ← LAST" if pos == n_updates - 1 else "")
        correct_tag = " ✓" if pos == n_updates - 1 else ""
        print(f"  v{pos:>2}: {count:>3}/{n_total} {bar}{tag}{correct_tag}")
    print(f"  garbage: {n_garbage}/{n_total}")